# Beacon: fresh Qwen3-4B multitask fine-tuning
17,446 training examples; 2,102 separate development examples. Predominantly English.
A new adapter on Qwen/Qwen3-4B-Instruct-2507. Previous Qwen2.5 adapters are never loaded.
Tasks: customer qualification facts, conversation classification, customer updates and demo selection. Application code calculates scores and verifies browser actions.
**Private notebook. Enable Internet and a GPU. Use Save Version > Save & Run All for a Kaggle server job that can continue with the laptop closed.**
Synthetic examples are not a guarantee of live accuracy. Test conversations are excluded. A time-limited partial run is labelled PARTIAL. The old Hugging Face model is never overwritten.


In [ ]:
import os,sys,json,hashlib,base64,lzma,subprocess,zipfile,time
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES']='0,1'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
WORK=Path('/kaggle/working/beacon_qwen3_fresh');WORK.mkdir(exist_ok=True)
sources=list(Path('/kaggle/input').rglob('training_source.json.xz'))
assert len(sources)==1, 'Attach the private Beacon Qwen3 training dataset containing training_source.json.xz'
payload=sources[0].read_bytes()
assert hashlib.sha256(payload).hexdigest()=='e88465cc6ea2c087ab1d9d76e981c559bf4843ea32e90fb2fd6cf64dfc61a7c5'
source_files=json.loads(lzma.decompress(payload))
source_manifest=json.loads(source_files['data/manifest.json'])
for relative,content in source_files.items():
    path=(WORK/relative).resolve()
    assert path.is_relative_to(WORK.resolve())
    raw=content.encode('utf-8')
    if relative in ('data/train.jsonl','data/dev.jsonl'):
        expected=source_manifest['splits'][Path(relative).stem]['sha256']
        if hashlib.sha256(raw).hexdigest()!=expected:
            # Text packaging normalizes Windows CRLF. Restore the original bytes,
            # and require the ORIGINAL hash; never silently change the manifest.
            raw=content.replace('\r\n','\n').replace('\n','\r\n').encode('utf-8')
        assert hashlib.sha256(raw).hexdigest()==expected, 'Dataset bytes failed verification'
    path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(raw)
os.chdir(WORK)
# Versioned trainer override: keep the uploaded data immutable and verify code separately.
trainer_source=lzma.decompress(base64.b64decode('/Td6WFoAAATm1rRGAgAhARYAAAB0L+Wj4CnED89dABFoDswmfBmIy3gkBf2AA+jQ7hmLMnuACgDzs3zN6XaCD9OR5mpMEccOzgMHSSY/ko305jNL5ZLbkeTw3w4sVSOjrYRpLHzS84t5gu9VNiDLelMhfbxatJisJ+wNX8RQHjNf72hJvpHPgaBubjL0JLFTkfvnkTO/Y1bu6eqIIPyCnc93nDEw8tXFh7wlBjXo57k0s6WA+CS88MYUKh08+d/Iq2H0dkxRGYDiu8DQ8Q+Pu28MwzA7ur47gAtxqH4kP0UA/YKTjwMK595AqVR5nzslVuDj1r8kWWpa6JzikWAdPbvNOo0kE2mi3vZGZqodY3E05CPYXrk5RsCn/MzpFMWpnF3cIjNgtTCLBad1ox/5Q6BU2gmmVVD4zbeTx/HWMs2yBU+ua77GM6XVutODyl+38wHTeqvaV5OiDVzGkt61G1eX0DjXQtMC5Ig0rRaNUavl7Pf5SwgFOnHqzx5KfcQ/XhUJM7FA8hZMzIQYL1vhQhEBKVQbYiFP/8iYtcD4dIMpwRSV21vLNneKNCxz4Q+UIKPNAszy0RxStABEMgwDxNHHh2TEfbYeG4DKRSY4Bzw6vvy/oiL7VVVMR6tLUlzvdIOB2Ph2T6PtMuRbP+8mdO8N3PDq5aKkD3/OG5/OiHfbSFgLxMPhFQtggaYip+p0hkhRinFsNGCK0NvO9Pnml+2Xmt8fLNi+s8540ZeuDWh65jADmtR0W0Sl934mmcpDfMK1hi07J2aLLt/sjy+pNcIULzMUVLdf1U/+SWd/dMkaKUgOWmYTnuXCKdS16tmME+x3bAr5jOIlcz7pftnBMzuuoTalBa0MEF54TBwrc7MnVJjckGgX9qvJBkC8127kiE+TE1oIZo0hz5gUn2ZX/EAO3O8S4OIOgYBa0rBsT2GkNZbGevB6zesyj4ryRCXiErd5aaH5bG9pyeZ4Pu2aEex2MwnIafB4+t9kr52SPm5sgWfZw4n7mLzkp9LEcgH07QCDhBc4tJVv5AL46dR98dUP30BKCI/N2NrVKos+KIyErcWfTAyd04YxWBhdwTplgpaa/8DvhA4BXmZG4CAZfFOeeMIqc8XX5Yxu9zNz7teCTetqP7CnPVuarU2quJk55B5YbT6fn9kx0RzzK/B68tJeDK9X7TEg6IDZL0mMVtGjhjBOTN4KuP8l8XEw1UhiYaDk28DuSHe6wZU5lG7IFApnJny22IPlG/PQROwvmdHAQfuiwcWcjtxSIsgwqe5Lt6dtXJMbHdH3gIaI+hDsF8bigQErCVi99pBvRiauMdTIKqE5pbPLlSaArpiNEuxBAF5nLCKll32YdOT1HcWqTuI1wxlEEQXZhlf9VsAoGMhaFIXcMr7vlUsvGPJWFav+CwR+mqLtNsLrvHZZM9OgKbLwpj/q2y1sNygURmS9xvTJAHDzHo9/JnyN7laaUmfynK2Njc1y+htL4wTPzzAIzvTNUxP+cmmydjGZewjUZ1Rs+N/Ouo+21crO76klVpI/bDvMhN3JE5njcbEhNmR8khLkMd8A0aebKlV9rAuWNx/J87YP2E6KY5Xu03hB4CdnSEhiCz7VnJxRZGiBC60v2QM33QSY4rloSZEUIFQ0+Gkzw7Nq4O+fxnMrq+pboLA02SlTZrhXsVCzEIcuTus1JSTrZnY9RlBqnxQw4OG8fR2DfydPTzfZRYxG1vuZakY/5uHi9JTGFZ6FD+K/y2YbpcMYD/peIA1etiZKNbgQlEQS4gAi2fejXW7SOQPWFFxC3zIaBups9mNnBm3VT6SNaZ0p2+d9hJt+vBGg0dv7LFh+MBunzzXU/22hVZbk/YkqtP551LWoCIeJbCZUR/vr6CILuV5naWKmvs4+kMXYFDRIrwPfiT7iN7PNXIR0sOYHexvznprEDs2xs1RIX7EAt9boDZmZWvXEG4gbj/IsR9HXlsp3BMv57Ts7xgGcPhcFXKEIRJLSk+ggtgv72SrMPaGUyjLxzeVVWMhyhNMNvdnqnjLC9asTg1/9tmISK22QpWDWbGWF4GdPp1j9pKpAS9sdpA0Q6+9vN5vhM8FNwBwLT4JoQUhGqOP4uBvKYH1mbaGyXyZFuJVeWt6vhUoEVMg9qsyQs72wSGnDHp7GVny2ELbc7QZkEIjvU4Hl251PQ8If+xySwzzvLFzQFmfusjf796mSy6lwzqwCeuOGb+8ymEdm3J5Rm7kQZ0MeoBed1hoYdslxQZcrdmW1Jo5FlM7ZFfg2U60F74IiFT0MqMSpUykiOm1vowc9MFrvvm5gZa9vKqRea/5pBmX/2Cc/WWG+rjUT42sD9scUX7ESUXQJy/MWaBUExjhI8hG/tkpLwrhhJICPcWjjeAtZCUiQgpq4UAHUKvYE2ZJDUQLzHPAL41iFqqm6/3sgpCO+uQQko33gfd/GfdXPux5Lq0wEHaDH3V33OdzyclteLM/cF0eVc4vxWKnQw4SOJtxS7JuH8VNnjdD785jS9psogWrJhMi0XQ+ms1kc2pV5uULQS6yqcH4dZ6baAVLHOO/I+ZLcmEXymvp0plmGLSqH37jdxSThTjvSu+pesFvp8q9PW+OgVSB80On1ZYMUtgMDF/XbNSpH/0hZ0AYalkHVgMQDIn7GpYk1360QgoxZ9wUPZ9bibvf+73B7SC02ncteX6DfaKrMiq0mwa4gFolDglwHy180x+xCwfhKDUscLvShxIi8GSXj4ywxpCM4xsg+WinKVUdaSAQ3Xg2L+ykDxOmceKFTZvO91Goz+k5qY9V12WKs8FdTcOqm8NGZeQlm/sp9I3X78aHt1d14VLPEncqkjZv8tiwmlkepO5NU+FjWRBA6RnSwNF8KZR52yIgRCYZ8YqaxYUP0TrBg4V7oEVcZib59Iwtje/VotY5il75KBO/xcGcD2un4WOhKZ2XQ99epIeiduHF5Fknx6ZMdVy6sUIkcxMOJeTnLmJgKBf0FlmLX6JCV86nw9JrBeWemYF1vNh3cebDC8S/we+10mN4y655mhKsju/XG2VxanShcdxY+yuBSqC0aznAe0d0+KhzN3zMNP4ze8aXJU6xfmXFSlD6WtC1k8WLxISvv/4f9gvqslsW+9TDa3h6XT6iDsCpUG5+7ooQwP9bj11jdDM/Y8+SZcKMvAHTBWyS3rt4P/7X3SgSNPU5eOXsxxnhG9wlxqpcobClYpl8kk5fxFv1DJE8J9A/V9oskiyOz6FDkfW2oxB5lxQfL/Nxu0ciRYAtdgIKGnPGLYPN2xO5RTvc49exZr7isqTrpL95uqLLz0Th3PXoJMDiLiC1nB4OvzvGHMuNBE1VyrRh7HsnPsV0EDJYlRPmxWcaSWOG3tUCAT1EqVFVlpsSAYuny0C3PGm9rQHNc//OChhb7zT73U+QSmMk9fMfq9NpRgzLCj1VXHrQbku1obYehE/sKdUxl4Qgokq9rI/2IOFB4RW/cv/+mvyToFTaxBchnCX7rxUrHP4warY5mvfUk6Q2LLOHSe+S6Nq4YFN6NVk4JObRnlTxJBcXNmOfjBOV7hWdWn0WeuckM7R5jbUI1+ixe7KwAsk86+hJMUUPofmyl4fHBskVGDwLtO31175wgFF0CxEnW840YUb4g7E0ZxKeBlP4OgJ23H9UQMa7yej5ChkzgTffPiquhap72WkDGibdRpO+M6KmeygMIkCOZHU2/jegAe/krOyWBhhKq8KNqI3FanPMuXoDh+0zszbNU9hBNiNP/bFNNJzygzxfve4w5Mfzk3WD5iE082xEv3FUrh2GTYyWEJD6XOhx080AoVsnGzpz17CkB349Lsdcw1tZlfG2+8iGJyXsgVA8bSHQZE2enxU5LfGU9k43BkXu7VxToX//D0d6L1oe/4xYe5mQVFmf8Xc9+Qtg9toKsv+l+UryQsWriCxq1Nzjle/ovfaaVXplJQIXGwAK5mxwpGTeElpUElYJ5tXjPDLi5tyY0d2qG1MglaO5MfkDJcAKLvtgWYMjHr8tR+teGKaybgDJk/ku+LhgY+bl49PAqP9TWWcl+8RelKDM0VffKsRGfi9WnxCmq4sHV5Z/6rJcegUjnhE6QkTgptSAMqAGdqvRfpopl3RxQze1t46aIeDmZDNKbHe/JUps3+Q2d3v0blNFKAnh26JL3Wxdb3HYcKMpXgGf9R8kB0czuL5nJdN8DUvVACgI7c+cdKZY6HmQ+tE6ZOBME6ckaXvhwWlv63fKK1ZpY9UYwLaQ1yXUzUdTFkVqLim/Ly8UnrBrdsW+wP6xuCIkRkrXZ83PBtLk0ozX18rGHKbnL1G5F6ftILPGGVID0vC6Y8l+0tMkN01z2BP16NHQuDZYh8FsZHzbTmFanOfsFFekuF6/iUcKi6FMoYdXzhIOHJFvRgGHUnx1UZGFnsFr9wZZsJ9P3vOhWaKVFZ4wxBVTk+i3YNEYZN12GBlabbGDGtZT1xar7PHjGSw33i5SukWo4bcmDdM4PsEK6I7Zzgo2xoBy9IDFG7eP110gtsPFXO8/PreCGfb9ASjOgUE4hCjximh9O8zgxh4F8IIJOByCu9vgT3BQb+ywRe/id1hiQ1VG0cCI3Equ2ooIr9yjTJ48efig5ZcZiSzaJS1Po7Vyg8PMHLBfajJRARhAC8tm+GJ3zsth6coq3vnbHf6KAh7A0tF0dwitZQpRXPzGOAEoOFrxkccZ4NrJg7Bhr5yf3J94Ip7hGhs1kBKl7BSaYV9bNHTYxRdKsiilNspMDkKDHkiF2l3rPdahwjnd4MHQjYPX7Mutn7cG/NGWqvKQ/C102NgMA9QrJSAY/E1yPzw+5Ee7VcR4S5LC7cGxepVdy92PzvgC+WOFTmH72v7jgRKXQ8oZ0ZgSzBktsQwu24LXrKQqte9JwR23MJlrWoLNeH/dFMpkMGf9PU3uBFEQzRL4apUs/QhH/0juH0NQQ4Roc7o+EY6XQedbEIxOEIKZvnHprUmxCpH1sYPChNBN2yxc8LpOr9O/BoEAyxBQBzpDVR1nHW9tWcwplACQ5C/0yQFKgmjJbzDamDnJEP/pVJjNomaAD2GTICj1TOJNyKQ1U441UZ5jY7t8K1opsNnB4Xys0SfmzTZZY2e92K1dl39ItPxGjOk2NOe/aUeBiaBlEm7Y7SGNq1WahBb/ARZsLEQSzO7GSRxuZOegfurJNAYyYAUveVVauGaPhKnAJwMIOI4aeVHhxOIgrRJb2AhLi/guk3jmhZvI4eEWPEKL2B1KFEh7FWwOEy94kAen9hDRfxRXn9GK+IiBOxO/xGudlMf9oWU2l5q96UPx3LlL8nOfBFczWZBF7NTl/Y9DVE/c83efV+jMGz1wjrvoJ9S5yAcIu1vvR49DN799LSCv8mD+7veRIiTBcuT6UAKFUhXK/lqEU2HqIJj8I4i7/Ay5MtV5j9abikjF5qNlMmFtO6AAAb8R0rrXcAggAAesfxVMAABKxD3KxxGf7AgAAAAAEWVo='))
assert hashlib.sha256(trainer_source).hexdigest()=='e009a599a95310881979595dc6587816a9985c54efd88ab655fa3cf0e0bc1855'
Path('train.py').write_bytes(trainer_source)
Path('trainer_provenance.json').write_text(json.dumps({'trainer_sha256':'e009a599a95310881979595dc6587816a9985c54efd88ab655fa3cf0e0bc1855','data_payload_sha256':hashlib.sha256(payload).hexdigest()}))
print('Verified trainer SHA256:','e009a599a95310881979595dc6587816a9985c54efd88ab655fa3cf0e0bc1855',flush=True)
def run(args):subprocess.run(args,check=True)
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','peft==0.17.1','accelerate==1.11.0','bitsandbytes==0.48.1','safetensors>=0.4.3'])
print('Verified fresh Qwen3 training package. No old adapter or test conversations included.',flush=True)


In [ ]:
run([sys.executable,'train.py','--check-only','--out','preflight'])
import torch
assert torch.cuda.is_available(), 'Select a Kaggle GPU before Save & Run All'
print('Kaggle GPU:',torch.cuda.get_device_name(0),flush=True)
assert torch.cuda.device_count()==2, 'Select GPU T4 x2; this notebook trains on both GPUs'


In [ ]:
# New LoRA weights on the pretrained Qwen3 base; never loads a previous adapter.
# Periodic checkpoints are saved. A 9-hour budget exports a PARTIAL result if needed.
run([sys.executable,'-m','torch.distributed.run','--standalone','--nproc_per_node=2','train.py','--out','run','--epochs','1','--lr','5e-5','--grad-accum','8','--wall-hours','9'])


In [ ]:
metadata=json.loads(Path('run/run_metadata.json').read_text())
assert metadata['base_model']=='Qwen/Qwen3-4B-Instruct-2507'
assert metadata['old_adapter_loaded'] is False
assert metadata['training_mode']=='new_adapter_on_pretrained_base'
name='beacon_qwen3_new_adapter.zip' if metadata['completed'] else 'beacon_qwen3_PARTIAL_adapter.zip'
output=Path('/kaggle/working')/name
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    z.write('trainer_provenance.json','trainer_provenance.json')
    for p in sorted(Path('run/adapter').rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to('run')))
    for p in ['run_metadata.json','data_manifest.json','preflight.json','trainer_state.json']:
        if Path('run',p).exists():z.write(Path('run',p),p)
with zipfile.ZipFile(output) as z:assert z.testzip() is None
print('COMPLETE' if metadata['completed'] else 'PARTIAL - NOT A COMPLETED TRAINING RUN',flush=True)
print('Download:',output,flush=True)
print('Evaluate on the separate local test set and live conversations before using this adapter in Beacon.')
